In [ ]:
def draw_lightning_overlay(year, hist, offset, lightning_files):

    lightning = extract_lightning(lightning_files[year])

    year_seconds = seconds_in_year(year)
    month_seconds = year_seconds / 12

    lightning_max = max(lightning)
    hist_max = hist.GetMaximum()

    scale = 1.2 * hist_max / lightning_max  

    g_lightning = ROOT.TGraph(12)

    for i, val in enumerate(lightning):

        x = offset + (i + 0.5) * month_seconds
        y = val * scale

        g_lightning.SetPoint(i, x, y)

    g_lightning.SetMarkerStyle(20)
    g_lightning.SetMarkerSize(1.0)
    g_lightning.SetMarkerColor(ROOT.kRed)
    g_lightning.SetLineColor(ROOT.kRed)

    g_lightning.Draw("P SAME")   # points only


    xmax = hist.GetXaxis().GetXmax()

    axis = ROOT.TGaxis( xmax, 0, xmax, hist_max, 0, lightning_max, 510, "+L")

    axis.SetLineColor(ROOT.kRed)
    axis.SetLabelColor(ROOT.kRed)
    axis.SetTitle("Lightning hours per month")

    axis.Draw()


    return g_lightning, axis

In [ ]:
#UoB Lightning

offset = 0
histograms = []
bins = 120


for i, year in enumerate(UoB_year_files):
    colour = colours[i]
    hist = year_hist_alongside(year, UoB_year_files[year], UNIX_year_start, colour, offset, UoB_total_span, bins)
    histograms.append((year, hist))
    offset += UoB_year_lengths[year]

ymax = 1.2 * max(h.GetMaximum() for _, h in histograms)


c_side = ROOT.TCanvas("c_side_by_side", "Years Side By Side", 1100, 500)

histograms[0][1].SetTitle("University of Birmingham Lightning Correlation")

first = True
for year, h in histograms:
    h.SetMaximum(ymax)
    h.SetMinimum(0)

    if first:
        h.Draw("HIST")
        first = False
    else:
        h.Draw("HIST SAME")

offset = 0
lightning_graphs = []

for year, h in histograms:
    g, axis = draw_lightning_overlay(year, h, offset, Bham_lightning_files)

    lightning_graphs.append(g)

    offset += UoB_year_lengths[year]

leg = ROOT.TLegend(0.1, 0.82, 0.9, 0.9)
leg.SetNColumns(len(histograms))

for year, h in histograms:
    leg.AddEntry(h, year, "l")

#leg.AddEntry(lightning_graphs[0], "Lightning", "p")

leg.Draw()

c_side.Draw()